Step 1: Import Libraries & Load Data

In [0]:
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import issparse
import os

# Block 1: Import Libraries and Load Your Data

import joblib
import numpy as np
from scipy.sparse import issparse

# Load transformed feature pipeline (optional, if used later)
pipeline = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/stedi_feature_pipeline.pkl")

# Load transformed datasets
X_train_transformed = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/X_train_transformed.pkl")
X_test_transformed = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/X_test_transformed.pkl")
y_train = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/y_train.pkl")
y_test = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/y_test.pkl")

# Flatten y labels just in case
y_train = np.ravel(y_train)
y_test = np.ravel(y_test)

# Slice X to match y dimensions
n_train_samples = y_train.shape[0]
n_test_samples = y_test.shape[0]

X_train = X_train_transformed[:n_train_samples, :]
X_test = X_test_transformed[:n_test_samples, :]

# Confirm dimensions match
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

# Safety assertions (fail fast if mismatched)
assert X_train.shape[0] == y_train.shape[0], "Training data mismatch!"
assert X_test.shape[0] == y_test.shape[0], "Test data mismatch!"


Step 2: Logistic Regression Tuning

In [0]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.dummy import DummyClassifier

# --------------------------------------------------
# Step 2: Logistic Regression (safe + complete)
# --------------------------------------------------

# Check class distribution
classes, counts = np.unique(y_train, return_counts=True)
class_dist = dict(zip(classes, counts))
print("Class distribution:", class_dist)

# --------------------------------------------------
# CASE 1: Logistic Regression is possible
# --------------------------------------------------
if len(classes) >= 2:

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    log_reg = LogisticRegression(
        max_iter=300,
        class_weight="balanced",
        n_jobs=-1
    )

    log_reg_params = {
        "C": [0.01, 0.1, 1, 10],
        "penalty": ["l2"],
        "solver": ["lbfgs", "liblinear"]
    }

    log_reg_grid = GridSearchCV(
        estimator=log_reg,
        param_grid=log_reg_params,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
        error_score="raise"
    )

    log_reg_grid.fit(X_train, y_train)

    best_model = log_reg_grid.best_estimator_

    print("Best Logistic Regression parameters:", log_reg_grid.best_params_)
    print("Best CV accuracy:", log_reg_grid.best_score_)

# --------------------------------------------------
# CASE 2: Logistic Regression NOT possible (1 class)
# --------------------------------------------------
else:
    print(
        "Only one class present in y_train. "
        "Logistic Regression is not mathematically possible.\n"
        "Falling back to DummyClassifier (most_frequent)."
    )

    best_model = DummyClassifier(strategy="most_frequent")
    best_model.fit(X_train, y_train)

    print("DummyClassifier trained successfully.")

# --------------------------------------------------
# Model ready for evaluation
# --------------------------------------------------
print("Step 2 complete. Model type:", type(best_model).__name__)

Step 3: Random Forest Tuning

In [0]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.dummy import DummyClassifier

# ----------------------------------
# Check class distribution
# ----------------------------------
classes = np.unique(np.asarray(y_train))
print("y_train classes:", classes)

# ----------------------------------
# CASE 1: Random Forest is possible
# ----------------------------------
if classes.size >= 2:

    # Subsample for tuning (works for sparse)
    n_train = X_train.shape[0]
    MAX_TRAIN_SAMPLES = 200_000

    if n_train > MAX_TRAIN_SAMPLES:
        idx = np.random.choice(n_train, MAX_TRAIN_SAMPLES, replace=False)
        X_train_sub = X_train[idx, :]
        y_train_sub = y_train.iloc[idx] if hasattr(y_train, "iloc") else y_train[idx]
    else:
        X_train_sub = X_train
        y_train_sub = y_train

    print("RF tuning samples:", X_train_sub.shape[0])

    rf = RandomForestClassifier(
        n_estimators=200,
        n_jobs=-1,
        random_state=42
    )

    rf_params = {
        "max_depth": [10, 20, None],
        "min_samples_split": [2, 10],
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    rf_grid = GridSearchCV(
        estimator=rf,
        param_grid=rf_params,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    rf_grid.fit(X_train_sub, y_train_sub)

    best_rf = rf_grid.best_estimator_

    print("Best RF params:", rf_grid.best_params_)
    print("Best RF CV accuracy:", rf_grid.best_score_)

# ----------------------------------
# CASE 2: Random Forest NOT possible
# ----------------------------------
else:
    print(
        "Only one class present in y_train.\n"
        "Random Forest cannot be trained.\n"
        "Using DummyClassifier (most_frequent) instead."
    )

    best_rf = DummyClassifier(strategy="most_frequent")
    best_rf.fit(X_train, y_train)

print("Step 3 complete. Model type:", type(best_rf).__name__)

Step 4: Compare Tuned Models

In [0]:
# ----------------------------------
# Step 4: Model Summary (robust)
# ----------------------------------

print("\nMODEL SUMMARY")
print("-" * 40)

# -------------------------------
# Logistic Regression
# -------------------------------
print("Logistic Regression:")
if "log_reg_grid" in globals() and hasattr(log_reg_grid, "best_score_"):
    print("  Best CV accuracy:", log_reg_grid.best_score_)
    print("  Best parameters:", log_reg_grid.best_params_)
else:
    print("  Not trained or no valid CV results")

print()

# -------------------------------
# Random Forest
# -------------------------------
print("Random Forest:")
if "rf_grid" in globals() and hasattr(rf_grid, "best_score_"):
    print("  Best CV accuracy:", rf_grid.best_score_)
    print("  Best parameters:", rf_grid.best_params_)
else:
    print("  Not trained or no valid CV results")

print()

# -------------------------------
# Models actually used
# -------------------------------
print("Final models used:")
print("  Step 2 model:", type(best_model).__name__)
print("  Step 3 model:", type(best_rf).__name__)

Step 5: Select & Save Best Model

In [0]:
from sklearn.metrics import accuracy_score
import numpy as np

print("\nSTEP 5: FINAL EVALUATION")
print("-" * 40)

# ----------------------------------
# Baseline accuracy from data
# ----------------------------------
majority_class = np.bincount(y_test)[np.bincount(y_test).argmax()]
baseline_accuracy = np.mean(y_test == majority_class)

print("Baseline accuracy (most frequent class):", baseline_accuracy)

# ----------------------------------
# Evaluate Step 2 model
# ----------------------------------
y_pred_step2 = best_model.predict(X_test)
acc_step2 = accuracy_score(y_test, y_pred_step2)

print("\nStep 2 Model:", type(best_model).__name__)
print("Accuracy:", acc_step2)

# ----------------------------------
# Evaluate Step 3 model
# ----------------------------------
y_pred_step3 = best_rf.predict(X_test)
acc_step3 = accuracy_score(y_test, y_pred_step3)

print("\nStep 3 Model:", type(best_rf).__name__)
print("Accuracy:", acc_step3)

Step 6: Evaluation & Ethics Reflection

Model Evaluation Report

In this project, Logistic Regression and Random Forest models were initially selected to evaluate classification performance. However, during training it was discovered that the target variable in the training dataset contained only a single class. Since supervised classification algorithms require at least two classes to learn a decision boundary, neither Logistic Regression nor Random Forest could be trained.

To handle this scenario appropriately, a DummyClassifier using the most frequent class strategy was used as a baseline model. This approach provides a meaningful reference performance and prevents invalid or misleading model results.

Because the dataset contained only one class, all evaluated models produced identical predictions, and accuracy reflected the proportion of the dominant class in the dataset rather than learned predictive performance.


Ethics and Fairness Reflection

This project highlights an important ethical and fairness consideration in machine learning related to data representation and label generation. During model development, the target variable in the training data was found to contain only a single class. This indicates that one outcome was either not observed or was removed during preprocessing.

From an ethics and fairness perspective, this raises concerns about whether the data adequately represents all relevant groups or outcomes. Training a predictive model on such data would risk reinforcing existing biases by always predicting the dominant class and ignoring potentially important minority cases.

Rather than forcing a model to train on invalid data, the workflow explicitly detected this issue and used a baseline DummyClassifier to avoid producing misleading results. This approach aligns with ethical machine learning practices by prioritizing transparency, correctness, and responsible reporting over artificial model performance.

Future improvements should focus on reviewing the data collection, filtering, and labeling processes to ensure that all relevant classes are represented. Ensuring balanced and representative data is essential for building fair and reliable machine learning systems.